# Model Comparison

Comparing models and choosing the best one.

In [0]:
%pip install feature_engine
%pip install xgboost lightgbm catboost
%pip install seaborn

In [0]:
import pandas as pd
import mlflow


In [0]:
experiment_id = 1203535636207065

max_results = 1000  # Maximum number of rows to retrieve
runs = mlflow.search_runs(experiment_ids=[experiment_id], max_results=max_results)

# Display dataframe in cell output
display(runs)

In [0]:
# Best Models Comparison & Leaderboard

# 1. Search all runs for this experiment (without IS NULL filter)
runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    order_by=["metrics.auc_test DESC"]
)

# 2. Filter out child runs in pandas
# GridSearchCV autolog creates child runs with 'tags.mlflow.parentRunId' populated
if "tags.mlflow.parentRunId" in runs.columns:
    runs = runs[runs["tags.mlflow.parentRunId"].isna()].copy()

# Filter only runs that have our logged test metric
if "metrics.auc_test" in runs.columns:
    runs = runs[runs["metrics.auc_test"].notna()].copy()

# 3. Select and rename relevant columns for comparison
columns_map = {
    'tags.mlflow.runName': 'Model Name',
    'metrics.best_cv_score': 'CV ROC-AUC',
    'metrics.auc_train': 'Train ROC-AUC',
    'metrics.auc_test': 'Test ROC-AUC',
    'metrics.recall_test_cutoff': 'Test Recall (Base Rate)',
    'metrics.recall_test_0_5': 'Test Recall (0.50)',
    'metrics.f1_test_cutoff': 'Test F1 (Base Rate)',
    'metrics.acc_test': 'Test Accuracy',
    'metrics.lift_auc_test_pct': 'Lift vs Random (%)',
    'params.base_rate_cutoff': 'Cutoff Threshold',
}

available_cols = [c for c in columns_map.keys() if c in runs.columns]
leaderboard = runs[available_cols].rename(columns=columns_map)

# 4. Sort and round numeric values
if 'Test ROC-AUC' in leaderboard.columns:
    leaderboard = leaderboard.sort_values(by='Test ROC-AUC', ascending=False).reset_index(drop=True)

float_cols = leaderboard.select_dtypes(include=['float64', 'float32']).columns
leaderboard[float_cols] = leaderboard[float_cols].round(4)

# 5. Display interactive Databricks table
display(leaderboard.drop_duplicates())

# Champion Model

## Best Models: Summary

**Key Insights for Decision-Making**

- Target Benchmark Achieved (PRD Section 3.2 & DEC-007):
    - The project target was Test ROC-AUC ≥ 0.83 and Recall ≥ 0.75.
    - XGBoost, Random Forest, LightGBM, and Logistic Regression all beat the ROC-AUC target (≥ 0.843) and easily exceed the 75% recall requirement at the base-rate cutoff (0.2669).

- The Champion Dilemma: XGBoost vs. LightGBM
    - churn_xgboost_tuned has the highest overall discriminative power (AUC: 0.8474) and the best balanced balance (F1: 0.6342, Accuracy: 80.12%), capturing 80.07% of churners with fewer false positives.
    - churn_lightgbm_tuned was trained with is_unbalance=True, which pushed its sensitivity to the extreme: it captures 91.36% of all churners (Recall), but at the cost of more false alarms (lower F1: 0.5735 and lower accuracy: 75.24%).

- Strong Linear Baseline:
    - churn_logistic_regression is remarkably strong (0.8437 Test AUC, virtually no overfitting), proving the feature engineering and imputation cascade are high quality and linearly informative.

- Models to Discard:
    - churn_mlp_tuned overfitted heavily (Train AUC 0.9697 → Test AUC 0.7917) and failed the PRD recall threshold (67.11% < 75%).
    - churn_knn and churn_decision_tree lag significantly behind the tree ensembles.

In [0]:
import mlflow

model_xgb_tunned = mlflow.sklearn.load_model("models:/workspace.default.churn_xgboost_tuned/1")
model_lightgbm_tunned = mlflow.sklearn.load_model("models:/workspace.default.churn_lightgbm_tuned/2")
model_lightgbm_balanced_tunned = mlflow.sklearn.load_model("models:/workspace.default.churn_lightgbm_balanced_tuned/1")

In [0]:
tb_abt2 = 'raw_abt2_Churn'

df = spark.table(f'workspace.default.{tb_abt2}').toPandas()


In [0]:
features = [
    # --- 1. Customer Demographics (Numeric Binary) ---
    'isMale',
    'SeniorCitizen',
    'isPartner',
    'HasDependents',
    'MonthlyIncome',

    # --- 2. Account & Tenure ---
    'nbrTenure',                  # Continuous tenure
    'isNewCustomer',              # High-risk early churn flag (tenure <= 6)
    'isPaperlessBilling',
    'isContractOneYear',          # Baseline = Month-to-month
    'isContractTwoYear',

    # --- 3. Payment Methods (One flag dropped as reference category) ---
    'isPaymentMethod_ElectronicCheck',  # Known highest-risk payment method
    'isPaymentMethod_MailedCheck',
    'isPaymentMethod_BankTransfer',
    'isPaymentMethod_CreditCard',

    # --- 4. Core Telecom Services ---
    'hasPhoneService',
    'hasMultipleLines',
    'hasInternetService',
    'isInternetService_DSL',
    'isInternetService_Fiber',          # Known high churn driver in EDA

    # --- 5. Add-on Internet Services ---
    'hasOnlineSecurity',
    'hasOnlineBackup',
    'hasDeviceProtection',
    'hasTechSupport',
    'hasStreamingTV',
    'hasStreamingMovies',
    'nbrInternetServicesAddOnQty',      # Total count of add-ons
    'hasSecurityBundle',                # High-loyalty flag

    # --- 6. High-Stakes Business Interactions (The Revenue Levers) ---
    'isM2M_dual',                       # The #1 revenue hemorrhage segment (47.3% churn)
    'isM2M_internetOnly',
    'isOneYear_dual',
    'isTwoYear_dual',

    # --- 7. Billing & Financial Ratios ---
    'MonthlyCharges',
    'TotalCharges',                     # Imputed by your champion Ridge model
    'TotalSpend',                       # Interaction (tenure * MonthlyCharges)
    'chargeRatio',                      # MonthlyCharges / MonthlyIncome (affordability stress)
    'numServices',                      # Service breadth
    'chargesPerService'                 # Value perception
]


In [0]:
from sklearn import model_selection

target = 'TargetChurn'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y,
                                                                    random_state=42,
                                                                    stratify=y,
                                                                    test_size=0.2)


print(f'Size Train dataset: {y_train.shape[0]} ({y_train.shape[0]/y.shape[0]*100:.2f}%). Target ratio: ({y_train.mean()*100:.2f}%)')

print(f'Size Test dataset: {y_test.shape[0]} ({y_test.shape[0]/y.shape[0]*100:.2f}%). Target ratio: ({y_test.mean()*100:.2f}%)')

## Feature Importance: XGB

In [0]:
# Feature Importance Analysis (XGBoost Champion)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# If using the pipeline returned by train_and_log_churn:
champion_pipeline = model_xgb_tunned

# Extract the classifier step from the sklearn pipeline
classifier = champion_pipeline.named_steps['classifier']

# 2. Extract feature importance values
# For XGBoost: feature_importances_ (gain-based)
importances = classifier.feature_importances_

df_importance = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

# Compute percentage contribution and cumulative sum
df_importance['Importance_Pct'] = (df_importance['Importance'] / df_importance['Importance'].sum()) * 100
df_importance['Cumulative_Pct'] = df_importance['Importance_Pct'].cumsum()

# Display interactive table in Databricks
display(df_importance.head(15))

# 3. Plot Top 15 Most Important Features
plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_importance.head(15),
    x='Importance_Pct',
    y='Feature',
    palette='Blues_r'
)
plt.title("Top 15 Feature Importances: Churn XGBoost Model", fontsize=14, fontweight='bold')
plt.xlabel("Relative Importance (%)", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.tight_layout()
plt.show()

In [0]:
# Threshold Tuning & Trade-Off Analysis
from sklearn import metrics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Predict raw churn probabilities on Test Set
y_test_probs = champion_pipeline.predict_proba(X_test[features])[:, 1]
base_rate = float(y_train.mean())  # ~0.2669

# 2. Evaluate metrics across thresholds from 0.05 to 0.90
thresholds = np.linspace(0.05, 0.90, 86)
records = []

for t in thresholds:
    y_pred_t = (y_test_probs >= t).astype(int)
    
    rec = metrics.recall_score(y_test, y_pred_t, zero_division=0)
    prec = metrics.precision_score(y_test, y_pred_t, zero_division=0)
    f1 = metrics.f1_score(y_test, y_pred_t, zero_division=0)
    acc = metrics.accuracy_score(y_test, y_pred_t)
    
    # % of total customers flagged for retention intervention
    pct_flagged = y_pred_t.mean() * 100
    
    records.append({
        'Threshold': round(t, 3),
        'Recall': round(rec, 4),
        'Precision': round(prec, 4),
        'F1_Score': round(f1, 4),
        'Accuracy': round(acc, 4),
        'Customers_Flagged_Pct': round(pct_flagged, 2)
    })

df_thresholds = pd.DataFrame(records)

# 3. Find key benchmark points
best_f1_row = df_thresholds.loc[df_thresholds['F1_Score'].idxmax()]
target_recall_row = df_thresholds[df_thresholds['Recall'] >= 0.75].sort_values(by='F1_Score', ascending=False).iloc[0]
base_rate_row = df_thresholds.iloc[(df_thresholds['Threshold'] - base_rate).abs().argsort()[:1]].iloc[0]

print("=== KEY THRESHOLD BENCHMARKS ===")
print(f"1. Base Rate Cutoff ({base_rate_row['Threshold']}): Recall = {base_rate_row['Recall']*100:.1f}%, Precision = {base_rate_row['Precision']*100:.1f}%, F1 = {base_rate_row['F1_Score']:.4f}")
print(f"2. Best F1 Cutoff ({best_f1_row['Threshold']}): Recall = {best_f1_row['Recall']*100:.1f}%, Precision = {best_f1_row['Precision']*100:.1f}%, F1 = {best_f1_row['F1_Score']:.4f}")
print(f"3. Max Precision for Recall >= 75% ({target_recall_row['Threshold']}): Recall = {target_recall_row['Recall']*100:.1f}%, Precision = {target_recall_row['Precision']*100:.1f}%, Flagged = {target_recall_row['Customers_Flagged_Pct']}%\n")

# 4. Plot Precision, Recall, and F1 vs. Threshold
plt.figure(figsize=(10, 6))
plt.plot(df_thresholds['Threshold'], df_thresholds['Recall'], label='Recall (Churn Detected)', color='navy', lw=2)
plt.plot(df_thresholds['Threshold'], df_thresholds['Precision'], label='Precision (True Churners in Outreach)', color='darkorange', lw=2)
plt.plot(df_thresholds['Threshold'], df_thresholds['F1_Score'], label='F1-Score (Harmonic Mean)', color='green', lw=2)

# Benchmark lines
plt.axvline(0.50, color='gray', linestyle=':', label='Default Cutoff (0.50)')
plt.axvline(base_rate, color='red', linestyle='--', label=f'Base Rate Cutoff ({base_rate:.2f})')
plt.axhline(0.75, color='black', linestyle='-.', alpha=0.6, label='PRD Target Recall (>= 0.75)')

plt.title("XGB: Threshold Tuning: Precision vs. Recall vs. F1", fontsize=14, fontweight='bold')
plt.xlabel("Probability Decision Threshold", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.xlim(0.05, 0.90)
plt.ylim(0, 1.02)
plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.25), ncol=3)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Light GBM (`is_unbalance=True`)

In [0]:
# Feature Importance Analysis (LightGBM Champion)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Retrieve the fitted pipeline

# If using the pipeline returned by train_and_log_churn:
champion_pipeline = model_lightgbm_tunned

# Extract the classifier step from the sklearn pipeline
classifier = champion_pipeline.named_steps['classifier']

# 2. Extract feature importance values
# For XGBoost: feature_importances_ (gain-based)
importances = classifier.feature_importances_

df_importance = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

# Compute percentage contribution and cumulative sum
df_importance['Importance_Pct'] = (df_importance['Importance'] / df_importance['Importance'].sum()) * 100
df_importance['Cumulative_Pct'] = df_importance['Importance_Pct'].cumsum()

# Display interactive table in Databricks
display(df_importance.head(15))

# 3. Plot Top 15 Most Important Features
plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_importance.head(15),
    x='Importance_Pct',
    y='Feature',
    palette='Blues_r'
)
plt.title("Top 15 Feature Importances: Churn Light GBM Model", fontsize=14, fontweight='bold')
plt.xlabel("Relative Importance (%)", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.tight_layout()
plt.show()

In [0]:
# Threshold Tuning & Trade-Off Analysis
from sklearn import metrics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Predict raw churn probabilities on Test Set
y_test_probs = champion_pipeline.predict_proba(X_test[features])[:, 1]
base_rate = float(y_train.mean())  # ~0.2669

# 2. Evaluate metrics across thresholds from 0.05 to 0.90
thresholds = np.linspace(0.05, 0.90, 86)
records = []

for t in thresholds:
    y_pred_t = (y_test_probs >= t).astype(int)
    
    rec = metrics.recall_score(y_test, y_pred_t, zero_division=0)
    prec = metrics.precision_score(y_test, y_pred_t, zero_division=0)
    f1 = metrics.f1_score(y_test, y_pred_t, zero_division=0)
    acc = metrics.accuracy_score(y_test, y_pred_t)
    
    # % of total customers flagged for retention intervention
    pct_flagged = y_pred_t.mean() * 100
    
    records.append({
        'Threshold': round(t, 3),
        'Recall': round(rec, 4),
        'Precision': round(prec, 4),
        'F1_Score': round(f1, 4),
        'Accuracy': round(acc, 4),
        'Customers_Flagged_Pct': round(pct_flagged, 2)
    })

df_thresholds = pd.DataFrame(records)

# 3. Find key benchmark points
best_f1_row = df_thresholds.loc[df_thresholds['F1_Score'].idxmax()]
target_recall_row = df_thresholds[df_thresholds['Recall'] >= 0.75].sort_values(by='F1_Score', ascending=False).iloc[0]
base_rate_row = df_thresholds.iloc[(df_thresholds['Threshold'] - base_rate).abs().argsort()[:1]].iloc[0]

print("=== KEY THRESHOLD BENCHMARKS ===")
print(f"1. Base Rate Cutoff ({base_rate_row['Threshold']}): Recall = {base_rate_row['Recall']*100:.1f}%, Precision = {base_rate_row['Precision']*100:.1f}%, F1 = {base_rate_row['F1_Score']:.4f}")
print(f"2. Best F1 Cutoff ({best_f1_row['Threshold']}): Recall = {best_f1_row['Recall']*100:.1f}%, Precision = {best_f1_row['Precision']*100:.1f}%, F1 = {best_f1_row['F1_Score']:.4f}")
print(f"3. Max Precision for Recall >= 75% ({target_recall_row['Threshold']}): Recall = {target_recall_row['Recall']*100:.1f}%, Precision = {target_recall_row['Precision']*100:.1f}%, Flagged = {target_recall_row['Customers_Flagged_Pct']}%\n")

# 4. Plot Precision, Recall, and F1 vs. Threshold
plt.figure(figsize=(10, 6))
plt.plot(df_thresholds['Threshold'], df_thresholds['Recall'], label='Recall (Churn Detected)', color='navy', lw=2)
plt.plot(df_thresholds['Threshold'], df_thresholds['Precision'], label='Precision (True Churners in Outreach)', color='darkorange', lw=2)
plt.plot(df_thresholds['Threshold'], df_thresholds['F1_Score'], label='F1-Score (Harmonic Mean)', color='green', lw=2)

# Benchmark lines
plt.axvline(0.50, color='gray', linestyle=':', label='Default Cutoff (0.50)')
plt.axvline(base_rate, color='red', linestyle='--', label=f'Base Rate Cutoff ({base_rate:.2f})')
plt.axhline(0.75, color='black', linestyle='-.', alpha=0.6, label='PRD Target Recall (>= 0.75)')

plt.title("Light GBM: Threshold Tuning: Precision vs. Recall vs. F1", fontsize=14, fontweight='bold')
plt.xlabel("Probability Decision Threshold", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.xlim(0.05, 0.90)
plt.ylim(0, 1.02)
plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.25), ncol=3)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Light GBM (`is_unbalance=False`)

In [0]:
# Feature Importance Analysis (LightGBM Champion)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Retrieve the fitted pipeline

# If using the pipeline returned by train_and_log_churn:
champion_pipeline = model_lightgbm_balanced_tunned

# Extract the classifier step from the sklearn pipeline
classifier = champion_pipeline.named_steps['classifier']

# 2. Extract feature importance values
# For XGBoost: feature_importances_ (gain-based)
importances = classifier.feature_importances_

df_importance = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

# Compute percentage contribution and cumulative sum
df_importance['Importance_Pct'] = (df_importance['Importance'] / df_importance['Importance'].sum()) * 100
df_importance['Cumulative_Pct'] = df_importance['Importance_Pct'].cumsum()

# Display interactive table in Databricks
display(df_importance.head(15))

# 3. Plot Top 15 Most Important Features
plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_importance.head(15),
    x='Importance_Pct',
    y='Feature',
    palette='Blues_r'
)
plt.title("Top 15 Feature Importances: Churn Light GBM (balanced) Model", fontsize=14, fontweight='bold')
plt.xlabel("Relative Importance (%)", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.tight_layout()
plt.show()

In [0]:
# Threshold Tuning & Trade-Off Analysis
from sklearn import metrics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Predict raw churn probabilities on Test Set
y_test_probs = champion_pipeline.predict_proba(X_test[features])[:, 1]
base_rate = float(y_train.mean())  # ~0.2669

# 2. Evaluate metrics across thresholds from 0.05 to 0.90
thresholds = np.linspace(0.05, 0.90, 86)
records = []

for t in thresholds:
    y_pred_t = (y_test_probs >= t).astype(int)
    
    rec = metrics.recall_score(y_test, y_pred_t, zero_division=0)
    prec = metrics.precision_score(y_test, y_pred_t, zero_division=0)
    f1 = metrics.f1_score(y_test, y_pred_t, zero_division=0)
    acc = metrics.accuracy_score(y_test, y_pred_t)
    
    # % of total customers flagged for retention intervention
    pct_flagged = y_pred_t.mean() * 100
    
    records.append({
        'Threshold': round(t, 3),
        'Recall': round(rec, 4),
        'Precision': round(prec, 4),
        'F1_Score': round(f1, 4),
        'Accuracy': round(acc, 4),
        'Customers_Flagged_Pct': round(pct_flagged, 2)
    })

df_thresholds = pd.DataFrame(records)

# 3. Find key benchmark points
best_f1_row = df_thresholds.loc[df_thresholds['F1_Score'].idxmax()]
target_recall_row = df_thresholds[df_thresholds['Recall'] >= 0.75].sort_values(by='F1_Score', ascending=False).iloc[0]
base_rate_row = df_thresholds.iloc[(df_thresholds['Threshold'] - base_rate).abs().argsort()[:1]].iloc[0]

print("=== KEY THRESHOLD BENCHMARKS ===")
print(f"1. Base Rate Cutoff ({base_rate_row['Threshold']}): Recall = {base_rate_row['Recall']*100:.1f}%, Precision = {base_rate_row['Precision']*100:.1f}%, F1 = {base_rate_row['F1_Score']:.4f}")
print(f"2. Best F1 Cutoff ({best_f1_row['Threshold']}): Recall = {best_f1_row['Recall']*100:.1f}%, Precision = {best_f1_row['Precision']*100:.1f}%, F1 = {best_f1_row['F1_Score']:.4f}")
print(f"3. Max Precision for Recall >= 75% ({target_recall_row['Threshold']}): Recall = {target_recall_row['Recall']*100:.1f}%, Precision = {target_recall_row['Precision']*100:.1f}%, Flagged = {target_recall_row['Customers_Flagged_Pct']}%\n")

# 4. Plot Precision, Recall, and F1 vs. Threshold
plt.figure(figsize=(10, 6))
plt.plot(df_thresholds['Threshold'], df_thresholds['Recall'], label='Recall (Churn Detected)', color='navy', lw=2)
plt.plot(df_thresholds['Threshold'], df_thresholds['Precision'], label='Precision (True Churners in Outreach)', color='darkorange', lw=2)
plt.plot(df_thresholds['Threshold'], df_thresholds['F1_Score'], label='F1-Score (Harmonic Mean)', color='green', lw=2)

# Benchmark lines
plt.axvline(0.50, color='gray', linestyle=':', label='Default Cutoff (0.50)')
plt.axvline(base_rate, color='red', linestyle='--', label=f'Base Rate Cutoff ({base_rate:.2f})')
plt.axhline(0.75, color='black', linestyle='-.', alpha=0.6, label='PRD Target Recall (>= 0.75)')

plt.title("Light GBM (balanced): Threshold Tuning: Precision vs. Recall vs. F1", fontsize=14, fontweight='bold')
plt.xlabel("Probability Decision Threshold", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.xlim(0.05, 0.90)
plt.ylim(0, 1.02)
plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.25), ncol=3)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Why is this my champion model?

After a modification on the Liight GBM `is_unbalance=False`.

In LightGBM, setting `is_unbalance=True` is a shortcut for handling imbalanced binary classification datasets. Its core function is to automatically adjust the weight of the minority class, preventing the model from ignoring the minority class due to its small sample size.

We have normally we consider a rate below 0.3 as low, but our data is ~27%, so we used this hyper parameter to check how the model would behave. It penalized too much on a churn ratio that was not severe.


### Light GBM (balanced)
This LightGBM has distributed importance (cumulative 52% across 5 features, top feature 13.3%), the healthy profile.

Criterion | XGBoost (churn_xgboost_tuned) | LightGBM balanced (churn_lightgbm_balanced_tuned) | Winner
--- | --- | --- | ---
CV ROC-AUC | 0.8383 | 0.8397 | LightGBM (marginal)
Test ROC-AUC | 0.8474 | 0.8442 | XGBoost (0.3pp,  noise)
Test Recall (base rate) | 0.8007 | 0.7973 | Tie
Test F1 (base rate) | 0.6342 | 0.6299 | Tie
Test Accuracy | 0.8012 | 0.8012 | Tie
Recall ≥ 75% precision | 0.552 | 0.549 | Tie (XGB +0.3pp)
Feature concentration | ~40% on isM2M_dual | 13.3% top feature, distributed | LightGBM
Robustness / explainability | Fragile, single-point-of-failure | Healthy | LightGBM




## Set Champion Model

In [0]:
from mlflow import MlflowClient

client = MlflowClient()

# Set alias 'champion' to Version 2 of your tuned model
client.set_registered_model_alias(
    name="workspace.default.churn_lightgbm_balanced_tuned",
    alias="champion",
    version="1"  
)

print("Champion alias successfully set!")

In [0]:
# Champion model
model = mlflow.sklearn.load_model("models:/workspace.default.churn_lightgbm_balanced_tuned@champion")

model